<a href="https://colab.research.google.com/github/ravikeerthi7606/Scrapyard_fire_smoke_detection/blob/main/Scrapyard_Fire_Smoke_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not enabled")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
%pip install -q kagglehub ultralytics opencv-python-headless pyyaml

In [7]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import kagglehub
from pathlib import Path

dataset_path = kagglehub.dataset_download(
    "sayedgamal99/smoke-fire-detection-yolo"
)

DATASET_DIR = Path(dataset_path)

print("Dataset location:", DATASET_DIR)
print("Dataset exists:", DATASET_DIR.exists())

for p in list(DATASET_DIR.rglob("*"))[:50]:
    print(p)

100%|██████████| 2.84G/2.84G [00:29<00:00, 104MB/s]

Extracting files...


Dataset location: /root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1
Dataset exists: True
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data.yaml
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/test
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/val
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/train
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/test/labels
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/test/images
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/val/labels
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/val/images
/root/.cache/kagglehub/dataset

In [4]:
import ultralytics
import cv2
import yaml
import os

print("Ultralytics:", ultralytics.__version__)
print("OpenCV:", cv2.__version__)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics: 8.4.197
OpenCV: 4.14.0


In [5]:
yaml_files = list(DATASET_DIR.rglob("*.yaml"))
yaml_files += list(DATASET_DIR.rglob("*.yml"))

print("YAML files:")
for p in yaml_files:
    print(p)

YAML files:
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data.yaml


In [6]:
import yaml

for p in yaml_files:
    print("\nFILE:", p)
    try:
        print(yaml.safe_load(p.read_text()))
    except Exception as e:
        print("Could not parse:", e)


FILE: /root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data.yaml
{'path': '/kaggle/working/D Fire Dataset', 'train': 'data/train/images', 'val': 'data/val/images', 'test': 'data/test/images', 'names': ['smoke', 'fire'], 'nc': 2, 'train_count': 14122, 'val_count': 3099, 'test_count': 4306}


In [8]:

# from pathlib import Path
# import zipfile

# ZIP_PATH = Path(
#     "/content/drive/MyDrive/scrapyard_project/D-Fire.zip"
# )

# EXTRACT_DIR = Path("/content/dfire_raw")

# assert ZIP_PATH.exists(), f"Dataset not found: {ZIP_PATH}"

# EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# with zipfile.ZipFile(ZIP_PATH) as z:
#     z.extractall(EXTRACT_DIR)

# print("Extraction completed")
# print("Top-level entries:", list(EXTRACT_DIR.iterdir())[:20])

In [9]:
from pathlib import Path

for p in DATASET_DIR.rglob("*"):
    if p.is_dir():
        print("DIR:", p)
# for p in list(EXTRACT_DIR.rglob("*"))[:60]:
#     print(p)

DIR: /root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data
DIR: /root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/test
DIR: /root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/val
DIR: /root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/train
DIR: /root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/test/labels
DIR: /root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/test/images
DIR: /root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/val/labels
DIR: /root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/val/images
DIR: /root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/train/labels
DIR: /root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/train/

In [10]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

all_images = [
    p for p in DATASET_DIR.rglob("*")
    if p.suffix.lower() in IMG_EXTS
]

all_labels = list(DATASET_DIR.rglob("*.txt"))

print("Image files:", len(all_images))
print("Text files:", len(all_labels))
# from pathlib import Path

# for p in EXTRACT_DIR.rglob("*"):
#     if p.is_dir():
#         print("DIR ", p)
#     elif p.suffix.lower() in {".yaml", ".yml", ".txt"}:
#         print("FILE", p)

Image files: 21527
Text files: 21527


In [28]:
from collections import Counter

counts = Counter()
for label_file in all_labels:
    for line in label_file.read_text().splitlines():
        if not line.strip():
            continue
        cls_id = int(float(line.split()[0]))
        counts[cls_id] += 1

print("Annotation counts:", counts)

# FIX: Point directories to correct nested 'data' subfolder
TRAIN_IMAGES = DATASET_DIR / "data/train/images"
TRAIN_LABELS = DATASET_DIR / "data/train/labels"

VAL_IMAGES = DATASET_DIR / "data/val/images"
VAL_LABELS = DATASET_DIR / "data/val/labels"

TEST_IMAGES = DATASET_DIR / "data/test/images"
TEST_LABELS = DATASET_DIR / "data/test/labels"

Annotation counts: Counter({1: 14692, 0: 11865})


In [30]:
for path in [
    TRAIN_IMAGES, TRAIN_LABELS,
    VAL_IMAGES, VAL_LABELS,
    TEST_IMAGES, TEST_LABELS
]:
    print(path, path.exists())

/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/train/images True
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/train/labels True
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/val/images True
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/val/labels True
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/test/images True
/root/.cache/kagglehub/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data/test/labels True


In [32]:
from pathlib import Path

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_images(folder):
    return sorted(
        p for p in Path(folder).rglob("*")
        if p.suffix.lower() in IMG_EXTS
    )

train_imgs = list_images(TRAIN_IMAGES)
val_imgs = list_images(VAL_IMAGES)
test_imgs = list_images(TEST_IMAGES)

print("Train:", len(train_imgs))
print("Validation:", len(val_imgs))
print("Test:", len(test_imgs))

# assert train_imgs and val_imgs and test_imgs

Train: 14122
Validation: 3099
Test: 4306


In [33]:
from collections import Counter

def count_classes(label_dir):
    counts = Counter()
    missing = 0
    bad = []

    for label_path in Path(label_dir).rglob("*.txt"):
        for line in label_path.read_text().splitlines():
            if not line.strip():
                continue

            parts = line.split()

            try:
                class_id = int(float(parts[0]))
                counts[class_id] += 1
            except (ValueError, IndexError):
                bad.append((label_path.name, line))

    return counts, bad

counts, bad = count_classes(TRAIN_LABELS)

print("Class counts:", counts)
print("Malformed annotations:", bad[:10])

Class counts: Counter({1: 9638, 0: 7794})
Malformed annotations: []


In [35]:
from pathlib import Path
import shutil
import yaml

OUT = Path("/content/scrapyard_dataset")

# Ensure clean directories
if OUT.exists():
    shutil.rmtree(OUT)

OUT.mkdir(parents=True, exist_ok=True)

SPLITS = {
    "train": (TRAIN_IMAGES, TRAIN_LABELS),
    "val": (VAL_IMAGES, VAL_LABELS),
    "test": (TEST_IMAGES, TEST_LABELS),
}

def prepare_split(split, img_dir, label_dir):
    dst_img = OUT / "images" / split
    dst_lbl = OUT / "labels" / split

    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    image_files = list_images(img_dir)
    copied = 0

    for img in image_files:
        label = Path(label_dir) / f"{img.stem}.txt"

        if not label.exists():
            candidates = list(Path(label_dir).rglob(img.stem + ".txt"))
            if candidates:
                label = candidates[0]

        if not label.exists():
            raise FileNotFoundError(
                f"Missing annotation for {img.name}"
            )

        converted = []
        for line in label.read_text().splitlines():
            if not line.strip():
                continue

            parts = line.split()
            if len(parts) != 5:
                raise ValueError(
                    f"Unexpected label format: {img.name}: {line}"
                )

            cls = int(parts[0])
            # Preserve original source class IDs (0: smoke, 1: fire)
            mapping = {0: 0, 1: 1}

            if cls not in mapping:
                raise ValueError(
                    f"Unexpected class {cls}: {img.name}"
                )

            converted.append(
                " ".join([str(mapping[cls])] + parts[1:])
            )

        shutil.copy2(img, dst_img / img.name)
        (dst_lbl / f"{img.stem}.txt").write_text(
            "\n".join(converted)
        )
        copied += 1

    print(split, "copied:", copied)

# Prepare splits
for split, (imgs, labels) in SPLITS.items():
    prepare_split(split, imgs, labels)

# Create data.yaml immediately inside this process
data_config = {
    "path": str(OUT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "smoke",
        1: "fire"
    }
}

yaml_path = OUT / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print("data.yaml successfully created inside scrapyard_dataset directory.")

train copied: 14122
val copied: 3099
test copied: 4306
data.yaml successfully created inside scrapyard_dataset directory.


In [25]:
import yaml

data_config = {
    "path": str(OUT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "smoke",
        1: "fire"
    }
}


yaml_path = OUT / "data.yaml"

with open(yaml_path, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print(yaml_path.read_text())

path: /content/scrapyard_dataset
train: images/train
val: images/val
test: images/test
names:
  0: smoke
  1: fire



In [26]:
import random
import cv2
import matplotlib.pyplot as plt

def visualize_sample(img_path):
    label_path = OUT / "labels" / "train" / f"{img_path.stem}.txt"

    img = cv2.imread(str(img_path))
    assert img is not None

    h, w = img.shape[:2]

    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue

        cls, xc, yc, bw, bh = map(float, line.split())

        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)

        cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)

        name = {0: "smoke", 1: "fire"}[int(cls)]
        cv2.putText(
            img, name, (x1, max(20, y1-5)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2
        )

    plt.figure(figsize=(10,6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

for img in random.sample(train_imgs, min(4, len(train_imgs))):
    visualize_sample(img)

In [21]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

In [27]:
from pathlib import Path
import yaml

with open(yaml_path, "r") as f:
    cfg = yaml.safe_load(f)

print("Names:", cfg["names"])

for split in ["train", "val", "test"]:
    img_dir = OUT / "images" / split
    lbl_dir = OUT / "labels" / split

    images = list_images(img_dir)
    labels = list(lbl_dir.rglob("*.txt"))

    print(
        split,
        "images:", len(images),
        "labels:", len(labels)
    )

assert cfg["names"] == {0: "smoke", 1: "fire"}

Names: {0: 'smoke', 1: 'fire'}
train images: 0 labels: 0
val images: 0 labels: 0
test images: 0 labels: 0


In [37]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data=str(yaml_path),
    epochs=10,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=2,
    patience=8,
    project="/content/runs/scrapyard",
    name="fire_smoke_yolo11n",
    pretrained=True,
    seed=42,
    plots=True
)

Ultralytics 8.4.197 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/scrapyard_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fire_smoke_yolo11n-4, nbs=64, nms=None, opset=None, 

In [42]:
best_model = YOLO(
    "/content/runs/scrapyard/fire_smoke_yolo11n-4/weights/best.pt"
)

metrics = best_model.val(
    data=str(yaml_path),
    split="test",
    imgsz=640,
    device=0
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Per-class mAP50:", metrics.box.maps)

Ultralytics 8.4.197 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,597,294 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 37.7±17.7 MB/s, size: 232.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/scrapyard_dataset/labels/test... 4306 images, 2005 backgrounds, 4 corrupt: 100% ━━━━━━━━━━━━ 4306/4306 627.5it/s 6.9s
val: /content/scrapyard_dataset/images/test/WEB10769.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.0296875]
val: /content/scrapyard_dataset/images/test/WEB10775.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.015625]
val: /content/scrapyard_dataset/images/test/WEB11243.jpg: corrupt JPEG restored and saved
val: /content/scrapyard_dataset/images/test/WEB11244.jpg: corrupt JPEG restore

In [43]:
pred_results = best_model.predict(
    source=str(OUT / "images" / "test"),
    conf=0.25,
    imgsz=640,
    save=True,
    max_det=100
)

print("Predictions completed")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/4306 /content/scrapyard_dataset/images/test/AoF06723.jpg: 384x640 (no detections), 8.1ms
image 2/4306 /content/scrapyard_dataset/images/test/AoF06724.jpg: 352x640 (no detections), 48.9ms
image 3/4306 /content/scrapyard_dataset/images/test/AoF06725.jpg: 384x640 (no detections), 11.6ms
image 4/4306 /content/scrapyard_dataset/images/test/AoF06726.jpg: 384x640 (no detections), 13.1ms
image 5/4306 /content/scrapyard_dataset/images/test/AoF06727.jpg: 38

In [45]:
from pathlib import Path
import shutil

drive_out = Path(
    "/content/runs/scrapyard/fire_smoke_yolo11n-4/weights"
)
drive_out.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    "/content/runs/scrapyard/fire_smoke_yolo11n-4/weights/best.pt",
    drive_out / "fire_smoke_best.pt"
)

print("Model saved to:", drive_out / "fire_smoke_best.pt")

Model saved to: /content/runs/scrapyard/fire_smoke_yolo11n-4/weights/fire_smoke_best.pt


In [ ]:
# # =====================================================================
# # 1. SETUP & INSTALLATION
# # =====================================================================
# !pip install -q kagglehub ultralytics opencv-python-headless pyyaml

# import os
# import yaml
# import random
# import shutil
# from pathlib import Path
# from collections import Counter
# import cv2
# import matplotlib.pyplot as plt
# import torch
# import kagglehub
# from ultralytics import YOLO
# from google.colab import drive

# # Check GPU Status
# print("PyTorch:", torch.__version__)
# print("CUDA available:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("GPU:", torch.cuda.get_device_name(0))
# else:
#     print("GPU not enabled")

# # Mount Google Drive
# drive.mount("/content/drive")

# # =====================================================================
# # 2. DATASET DOWNLOADING & INSPECTION
# # =====================================================================
# dataset_path = kagglehub.dataset_download("sayedgamal99/smoke-fire-detection-yolo")
# DATASET_DIR = Path(dataset_path)
# print("Dataset location:", DATASET_DIR)
# print("Dataset exists:", DATASET_DIR.exists())

# # Define the input split paths from the downloaded Kaggle dataset
# TRAIN_IMAGES = DATASET_DIR / "train/images"
# TRAIN_LABELS = DATASET_DIR / "train/labels"
# VAL_IMAGES = DATASET_DIR / "valid/images"
# VAL_LABELS = DATASET_DIR / "valid/labels"
# TEST_IMAGES = DATASET_DIR / "test/images"
# TEST_LABELS = DATASET_DIR / "test/labels"

# # Verify path existence
# for path in [TRAIN_IMAGES, TRAIN_LABELS, VAL_IMAGES, VAL_LABELS, TEST_IMAGES, TEST_LABELS]:
#     print(path, "Exists:", path.exists())

# # =====================================================================
# # 3. PREPARE STRUCTURED DATASET & MAP CLASSES
# # =====================================================================
# OUT = Path("/content/scrapyard_dataset")
# SPLITS = {
#     "train": (TRAIN_IMAGES, TRAIN_LABELS),
#     "val": (VAL_IMAGES, VAL_LABELS),
#     "test": (TEST_IMAGES, TEST_LABELS),
# }

# IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# def list_images(folder):
#     return sorted(
#         p for p in Path(folder).rglob("*")
#         if p.suffix.lower() in IMG_EXTS
#     )

# def prepare_split(split, img_dir, label_dir):
#     dst_img = OUT / "images" / split
#     dst_lbl = OUT / "labels" / split

#     dst_img.mkdir(parents=True, exist_ok=True)
#     dst_lbl.mkdir(parents=True, exist_ok=True)

#     image_files = list_images(img_dir)
#     copied = 0

#     for img in image_files:
#         label = Path(label_dir) / f"{img.stem}.txt"

#         if not label.exists():
#             candidates = list(Path(label_dir).rglob(img.stem + ".txt"))
#             if candidates:
#                 label = candidates[0]

#         if not label.exists():
#             raise FileNotFoundError(f"Missing annotation for {img.name}")

#         converted = []
#         for line in label.read_text().splitlines():
#             if not line.strip():
#                 continue

#             parts = line.split()
#             if len(parts) != 5:
#                 raise ValueError(f"Unexpected label format: {img.name}: {line}")

#             cls = int(parts[0])
#             # Source: 0=smoke, 1=fire -> Target: 0=fire, 1=smoke
#             mapping = {0: 1, 1: 0}

#             if cls not in mapping:
#                 raise ValueError(f"Unexpected class {cls}: {img.name}")

#             converted.append(" ".join([str(mapping[cls])] + parts[1:]))

#         shutil.copy2(img, dst_img / img.name)
#         (dst_lbl / f"{img.stem}.txt").write_text("\n".join(converted))
#         copied += 1

#     print(f"Split {split} completed. Copied {copied} files.")

# # Create directory and process the dataset
# for split, (imgs, labels) in SPLITS.items():
#     prepare_split(split, imgs, labels)

# # =====================================================================
# # 4. CREATE DATA CONFIG YAML
# # =====================================================================
# data_config = {
#     "path": str(OUT),
#     "train": "images/train",
#     "val": "images/val",
#     "test": "images/test",
#     "names": {
#         0: "fire",
#         1: "smoke"
#     }
# }

# yaml_path = OUT / "data.yaml"
# with open(yaml_path, "w") as f:
#     yaml.safe_dump(data_config, f, sort_keys=False)

# print("\nYAML Config file created:\n", yaml_path.read_text())

# # =====================================================================
# # 5. MODEL TRAINING
# # =====================================================================
# model = YOLO("yolo11n.pt")

# results = model.train(
#     data=str(yaml_path),
#     epochs=30,
#     imgsz=640,
#     batch=16,
#     device=0 if torch.cuda.is_available() else "cpu",
#     workers=2,
#     patience=8,
#     project="/content/runs/scrapyard",
#     name="fire_smoke_yolo11n",
#     pretrained=True,
#     seed=42,
#     plots=True
# )

# # =====================================================================
# # 6. EVALUATION & PREDICTION ON TEST SET
# # =====================================================================
# best_model = YOLO("/content/runs/scrapyard/fire_smoke_yolo11n/weights/best.pt")

# metrics = best_model.val(
#     data=str(yaml_path),
#     split="test",
#     imgsz=640,
#     device=0 if torch.cuda.is_available() else "cpu"
# )

# print("mAP50:", metrics.box.map50)
# print("mAP50-95:", metrics.box.map)
# print("Per-class mAP50:", metrics.box.maps)

# pred_results = best_model.predict(
#     source=str(OUT / "images" / "test"),
#     conf=0.25,
#     imgsz=640,
#     save=True,
#     max_det=100
# )
# print("Predictions completed on test set.")

# # =====================================================================
# # 7. EXPORT BEST WEIGHTS TO GOOGLE DRIVE
# # =====================================================================
# drive_out = Path("/content/drive/MyDrive/scrapyard_project/weights")
# drive_out.mkdir(parents=True, exist_ok=True)

# shutil.copy2(
#     "/content/runs/scrapyard/fire_smoke_yolo11n/weights/best.pt",
#     drive_out / "fire_smoke_best.pt"
# )

# print("Model weights successfully saved to Google Drive at:", drive_out / "fire_smoke_best.pt")